# TIPy Double Inverted Pendulum — DQN + MJX GPU

This notebook is the **native GPU version** of DQN for the double-inverted-pendulum-on-a-cart task.

DQN uses discrete actions, so we tile the force range into `NUM_ACTIONS = 9` evenly-spaced values and learn a Q-function over them.

The physics collection path:

```text
batched MJX physics (NUM_ENVS parallel worlds)
          ↓
batched observations
          ↓
epsilon-greedy / greedy Q-network
          ↓
discrete action → force lookup
          ↓
batched MJX physics
```

The per-step collection uses `jax.vmap + jax.jit`. The replay buffer lives on CPU as numpy arrays (random indexing is simpler on CPU); sampled mini-batches are pushed to the GPU for the network update.

### Task

- Double inverted pendulum on a cart
- Cart usable range: **-1.5 m to +1.5 m**
- Discrete force: **9 levels from -40 N to +40 N**
- Swing up both links, capture upright, balance
- Double DQN with a 3-hidden-layer Q-network and hard target updates


## 1. Colab / GPU setup

MJX is a separate package from normal MuJoCo. The cell installs the JAX implementation, confirms CUDA, and mounts Drive for checkpoints.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Recommended by the current MJX docs for NVIDIA GPU performance.
os.environ["XLA_FLAGS"] = (
    os.environ.get("XLA_FLAGS", "")
    + " --xla_gpu_triton_gemm_any=true"
)

if IN_COLAB or IN_KAGGLE:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--upgrade", "-q",
            "jax[cuda12]",
            "mujoco-mjx",
            "flax==0.12.8",
            "optax==0.2.8",
            "pandas>=2.0",
            "matplotlib>=3.8",
            "imageio>=2.34",
            "imageio-ffmpeg>=0.5",
        ],
        check=True,
    )

import jax
import jax.numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
import flax
import flax.linen as nn
import optax

print("JAX:", jax.__version__)
print("MuJoCo:", mujoco.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())

gpu_devices = [d for d in jax.devices() if d.platform == "gpu"]
if (IN_COLAB or IN_KAGGLE) and not gpu_devices:
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime → Change runtime type → GPU, "
        "restart the runtime, then run from the top."
    )


## 2. Training configuration

DQN does not need as many parallel environments as PPO (the replay buffer decouples collection from learning). `NUM_ENVS = 512` still gives substantial GPU utilisation for the batched MJX physics step.

The replay buffer (`BUFFER_CAPACITY`) is stored on CPU as numpy arrays. Each entry is one `(obs, action, reward, next_obs, terminal)` tuple from a single environment step. With `NUM_ENVS` transitions added every Python iteration and CPU sampling, the memory footprint stays predictable.


In [ ]:
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/double/dqn-mjx/run-001")
    if IN_COLAB
    else Path.cwd() / "double-dqn-mjx-run"
)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH = OUTPUT_DIR / "dashboard.png"
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"

SEED = 42

# Physics / task
MAX_EPISODE_STEPS = 5000
ACTION_LIMIT = 40.0      # N — lighter reference system
RAIL_LIMIT = 1.5         # m

# MJX batch
NUM_ENVS = 512           # DQN doesn't need as many parallel envs as PPO
TOTAL_STEPS = 5_000_000

# DQN hyperparameters
LEARNING_RATE = 3e-4
GAMMA = 0.99
BATCH_SIZE = 256
BUFFER_CAPACITY = 200_000   # flat ring buffer stored as numpy arrays on CPU
WARMUP_STEPS = 2_000        # global steps before first update
TRAIN_FREQUENCY = 4         # update every N global steps
TARGET_FREQUENCY = 2_000    # hard target network copy

# Action discretization: 9 discrete forces spanning [-ACTION_LIMIT, +ACTION_LIMIT]
NUM_ACTIONS = 9

# Epsilon-greedy
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY_STEPS = int(TOTAL_STEPS * 0.6)

# Logging
CHECKPOINT_EVERY_STEPS = 50_000
LOG_EVERY_STEPS = 5_000

SMOKE_TEST = False
if SMOKE_TEST:
    TOTAL_STEPS = 5_000
    WARMUP_STEPS = 256
    BUFFER_CAPACITY = 2_000
    CHECKPOINT_EVERY_STEPS = 2_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("output:", OUTPUT_DIR)


## 3. MuJoCo model

This notebook uses the **reference physical parameters** from the paper:

| Parameter | Value |
|---|---|
| Cart mass | 0.4 kg |
| Pole 1 mass | 0.15 kg, L = 0.5 m |
| Pole 2 mass | 0.15 kg, L = 0.5 m |
| J₁ = J₂ | m·L²/12 ≈ 0.003125 kg·m² |
| Cart damping | 0.05 |
| Hinge damping | 0.01 (numerical stability only) |
| Actuator limit | ±40 N |

The lighter system (vs. the PPO notebook) is intentionally easier to balance — DQN's discrete action space is a harder inductive bias than a continuous Gaussian policy, so matching physics difficulty matters.

Two MJX-specific choices:
1. All geoms have `contype/conaffinity=0` — no contact computation.
2. `solver="Newton" iterations="1" ls_iterations="2" jacobian="dense"` — MJX-optimised settings from the MJX documentation.


In [ ]:
MODEL_XML = r"""
<mujoco model="cartpole_double_dqn">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option
      timestep="0.005"
      gravity="0 0 -9.81"
      integrator="RK4"
      solver="Newton"
      iterations="1"
      ls_iterations="2"
      jacobian="dense">
    <flag eulerdamp="disable"/>
  </option>

  <default>
    <geom contype="0" conaffinity="0" />
  </default>

  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>

  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole1_mat" rgba="0.2 0.75 0.3 1" />
    <material name="pole2_mat" rgba="0.2 0.3 0.75 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>

  <worldbody>
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />

    <body name="frame">
      <geom name="tower_left" type="box" pos="-1.8 0 0.4" size="0.08 0.12 0.8" material="metal_mat" />
      <geom name="tower_right" type="box" pos="1.8 0 0.4" size="0.08 0.12 0.8" material="metal_mat" />
      <geom name="rail" type="box" pos="0 0 0.85" size="1.5 0.04 0.04" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-1.5 0 0.95" size="0.02 0.1 0.08" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="1.5 0 0.95" size="0.02 0.1 0.08" material="rail_limit_mat" />
    </body>

    <body name="cart" pos="0 0 1">
      <joint
          name="cart_slide"
          type="slide"
          axis="1 0 0"
          limited="false"
          frictionloss="0"
          damping="0.05" />
      <inertial pos="0 0 0" mass="0.4" diaginertia="0.00533 0.00533 0.00533" />
      <geom name="cart_geom" type="box" size="0.07 0.065 0.08" mass="0.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.08 0"
            axisangle="1 0 0 1.5708" size="0.012 0.025" material="metal_mat" />

      <!-- pole1: thin rod, mass=0.15 kg, L=0.5 m, starts hanging down (ref=pi) -->
      <!-- Body offset 0.25 m places the inertial COM at the rod midpoint       -->
      <body name="pole1" pos="0 0.09 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole1_hinge" type="hinge" axis="0 -1 0"
               frictionloss="0" damping="0.01"
               ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.25" mass="0.15"
                  diaginertia="0.0125 0.0125 1e-5" />
        <site name="pole1_hinge_site" pos="0 0 0" size="0.012" type="sphere" material="site_mat" />
        <geom name="pole1_geom" type="capsule" pos="0 0 0.25"
              size="0.01 0.25" mass="0.0" material="pole1_mat" />
        <site name="pole1_tip_site" pos="0 0 0.5" size="0.012" type="sphere" material="site_mat" />
        <geom name="pole2_mount_pin" type="capsule" pos="0 0.012 0.5"
              axisangle="1 0 0 1.5708" size="0.01 0.012" material="metal_mat" />

        <!-- pole2: same masses/dims, relative joint ref=0 -->
        <body name="pole2" pos="0 0.025 0.5" quat="1 0 0 0">
          <joint name="pole2_hinge" type="hinge" axis="0 -1 0"
                 frictionloss="0" damping="0.01" ref="0" limited="false" />
          <inertial pos="0 0 0.25" mass="0.15"
                    diaginertia="0.0125 0.0125 1e-5" />
          <site name="pole2_hinge_site" pos="0 0 0" size="0.012" type="sphere" material="site_mat" />
          <geom name="pole2_geom" type="capsule" pos="0 0 0.25"
                size="0.01 0.25" mass="0.0" material="pole2_mat" />
          <site name="pole2_tip_site" pos="0 0 0.5" size="0.012" type="sphere" material="site_mat" />
        </body>
      </body>
    </body>

    <camera name="replay" pos="0 5 1.4" fovy="50"
            xyaxes="-1 0 0 0 -0.15 0.988686" />

  </worldbody>

  <actuator>
    <motor name="cart_motor" joint="cart_slide"
           gear="1" ctrlrange="-40 40" forcerange="-40 40" />
  </actuator>
</mujoco>
"""

mj_model = mujoco.MjModel.from_xml_string(MODEL_XML)

# Pure JAX MJX implementation.
mjx_model = mjx.put_model(mj_model, impl="jax")

print("nq:", mj_model.nq, "nv:", mj_model.nv, "nu:", mj_model.nu)
print("timestep:", mj_model.opt.timestep)


## 4. Pure-JAX batched environment

There is no `gym.Env` class here. Environment state is explicit JAX data stored as an `EnvState` struct dataclass (traceable by JAX).

### Observation (8-D)

```text
0  clip(x, -1.5, 1.5)        cart position
1  clip(dx / 3.0, -1, 1)     cart velocity (normalised for this lighter system)
2  cos(theta1)               absolute angle of pole 1 from vertical
3  sin(theta1)
4  cos(theta2)               absolute angle of pole 2 from vertical
5  sin(theta2)
6  clip(dtheta1 / 15, -1, 1) pole-1 angular velocity
7  clip(dtheta2 / 15, -1, 1) pole-2 angular velocity
```

`theta1` and `theta2` are **absolute** angles from the upright vertical, computed from the MJX `qpos` which stores relative joint angles:

```
theta1 = wrap(qpos[1])              # pole1 relative joint is also absolute here
theta2 = wrap(qpos[1] + qpos[2])    # pole2 absolute = pole1 + relative
```

### Action table

```python
ACTION_TABLE = jnp.linspace(-ACTION_LIMIT, +ACTION_LIMIT, NUM_ACTIONS)
# → [-40, -30, -20, -10, 0, +10, +20, +30, +40] N
```

### Reset

Small random perturbation around the hanging-down equilibrium (both poles at π from vertical) with ±0.05 rad, ±0.05 rad/s, ±0.05 m.


In [ ]:
from flax import struct

OBS_DIM = 8

# Discrete force lookup table — index i → force in Newtons.
ACTION_TABLE = jnp.linspace(-ACTION_LIMIT, ACTION_LIMIT, NUM_ACTIONS)

@struct.dataclass
class EnvState:
    data: object
    step_count: jax.Array
    episode_return: jax.Array
    episode_length: jax.Array
    captured_upright: jax.Array


def wrap_angle(x):
    """Map any angle to (-pi, pi]."""
    return (x + jnp.pi) % (2.0 * jnp.pi) - jnp.pi


def physical_state(data):
    """Extract physical quantities from MJX data."""
    x = data.qpos[..., 0]
    relative1 = data.qpos[..., 1]
    relative2 = data.qpos[..., 2]

    dx = data.qvel[..., 0]
    relative_speed1 = data.qvel[..., 1]
    relative_speed2 = data.qvel[..., 2]

    # Absolute angles from vertical (0 = upright).
    theta1 = wrap_angle(relative1)
    theta2 = wrap_angle(relative1 + relative2)

    # Absolute angular velocities.
    dtheta1 = relative_speed1
    dtheta2 = relative_speed1 + relative_speed2

    return x, theta1, theta2, dx, dtheta1, dtheta2


def observation_from_data(data):
    x, theta1, theta2, dx, dtheta1, dtheta2 = physical_state(data)

    return jnp.stack(
        [
            jnp.clip(x, -RAIL_LIMIT, RAIL_LIMIT),
            jnp.clip(dx / 3.0, -1.0, 1.0),
            jnp.cos(theta1),
            jnp.sin(theta1),
            jnp.cos(theta2),
            jnp.sin(theta2),
            jnp.clip(dtheta1 / 15.0, -1.0, 1.0),
            jnp.clip(dtheta2 / 15.0, -1.0, 1.0),
        ],
        axis=-1,
    ).astype(jnp.float32)


def _reset_one(key):
    """Reset a single environment to a small random perturbation around hanging-down."""
    key_pos, key_ang1, key_ang2, key_vel, key_w1, key_w2 = jax.random.split(key, 6)

    data = mjx.make_data(mjx_model)

    # Absolute angles around hanging-down (pi from upright).
    absolute1 = jnp.pi + jax.random.uniform(
        key_ang1, (), minval=-0.05, maxval=0.05
    )
    absolute2 = jnp.pi + jax.random.uniform(
        key_ang2, (), minval=-0.05, maxval=0.05
    )

    # MJX stores qpos[1] as pole1 joint angle, qpos[2] as pole2 RELATIVE joint.
    qpos = jnp.array(
        [
            jax.random.uniform(key_pos, (), minval=-0.05, maxval=0.05),
            absolute1,
            absolute2 - absolute1,   # relative angle for pole2 joint
        ],
        dtype=jnp.float32,
    )

    absolute_speed1 = jax.random.uniform(
        key_w1, (), minval=-0.05, maxval=0.05
    )
    absolute_speed2 = jax.random.uniform(
        key_w2, (), minval=-0.05, maxval=0.05
    )

    qvel = jnp.array(
        [
            jax.random.uniform(key_vel, (), minval=-0.05, maxval=0.05),
            absolute_speed1,
            absolute_speed2 - absolute_speed1,   # relative angular speed
        ],
        dtype=jnp.float32,
    )

    data = data.replace(qpos=qpos, qvel=qvel)
    data = mjx.forward(mjx_model, data)
    return data


reset_data_batch = jax.jit(jax.vmap(_reset_one))


def reset_batch(keys):
    data = reset_data_batch(keys)
    n = keys.shape[0]
    state = EnvState(
        data=data,
        step_count=jnp.zeros((n,), dtype=jnp.int32),
        episode_return=jnp.zeros((n,), dtype=jnp.float32),
        episode_length=jnp.zeros((n,), dtype=jnp.int32),
        captured_upright=jnp.zeros((n,), dtype=jnp.bool_),
    )
    return state, observation_from_data(data)


def _step_one(data, action_idx):
    """Step a single MJX world with a discrete action index."""
    force = ACTION_TABLE[action_idx]
    ctrl = data.ctrl.at[0].set(force)
    data = data.replace(ctrl=ctrl)
    data = mjx.step(mjx_model, data)
    return data


step_data_batch = jax.vmap(_step_one)


def _compute_reward(data):
    """Reward shaping identical to the PPO notebook for consistency."""
    x, theta1, theta2, dx, dtheta1, dtheta2 = physical_state(data)

    angle_reward = 0.5 * (jnp.cos(theta1) + jnp.cos(theta2))
    position_penalty = 0.2 * (x / RAIL_LIMIT) ** 2
    velocity_penalty = 0.005 * dx ** 2
    ang_vel_penalty = 0.002 * (dtheta1 ** 2 + dtheta2 ** 2)
    upright_bonus = jnp.where(
        (jnp.abs(theta1) < 0.3) & (jnp.abs(theta2) < 0.3), 2.0, 0.0
    )

    reward = (
        angle_reward
        - position_penalty
        - velocity_penalty
        - ang_vel_penalty
        + upright_bonus
    )

    terminated = jnp.abs(x) >= RAIL_LIMIT
    reward = reward - jnp.where(terminated, 50.0, 0.0)

    return reward, terminated, x, theta1, theta2


## 5. Replay buffer (CPU numpy ring buffer)

The buffer is intentionally on CPU. JAX's `jnp.take` with dynamically-computed random indices would require `jax.lax.dynamic_slice` and shape-static indexing, which is harder to express than a simple numpy fancy-index. The CPU→GPU transfer for a mini-batch of 256 observations is negligible compared to the Q-network forward/backward pass.

**Design**:
- Flat ring of `BUFFER_CAPACITY` transitions.
- `add_batch` writes `NUM_ENVS` rows per call using modular indexing.
- `sample` draws `BATCH_SIZE` rows with `np.random.choice` (no replacement).


In [ ]:
class ReplayBuffer:
    """Flat CPU ring buffer for DQN experience replay."""

    def __init__(self, capacity: int, obs_dim: int):
        self.capacity = capacity
        self.obs_dim = obs_dim
        self._obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self._actions = np.zeros(capacity, dtype=np.int32)
        self._rewards = np.zeros(capacity, dtype=np.float32)
        self._next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self._terminals = np.zeros(capacity, dtype=np.bool_)
        self._pointer = 0
        self.size = 0

    def add_batch(
        self,
        obs: np.ndarray,         # (N, obs_dim)
        actions: np.ndarray,     # (N,)
        rewards: np.ndarray,     # (N,)
        next_obs: np.ndarray,    # (N, obs_dim)
        terminals: np.ndarray,   # (N,)
    ) -> None:
        n = obs.shape[0]
        indices = (self._pointer + np.arange(n)) % self.capacity
        self._obs[indices] = obs
        self._actions[indices] = actions
        self._rewards[indices] = rewards
        self._next_obs[indices] = next_obs
        self._terminals[indices] = terminals
        self._pointer = (self._pointer + n) % self.capacity
        self.size = min(self.size + n, self.capacity)

    def sample(self, batch_size: int) -> dict:
        indices = np.random.choice(self.size, batch_size, replace=False)
        return {
            "obs": self._obs[indices],
            "actions": self._actions[indices],
            "rewards": self._rewards[indices],
            "next_obs": self._next_obs[indices],
            "terminals": self._terminals[indices].astype(np.float32),
        }


replay_buffer = ReplayBuffer(BUFFER_CAPACITY, OBS_DIM)
print(f"Replay buffer: capacity={BUFFER_CAPACITY:,}, obs_dim={OBS_DIM}")


## 6. Q-Network (Flax)

Three hidden layers (256 → 256 → 128) match the representational capacity of the PPO actor-critic. The output is a vector of Q-values — one per discrete action.

Orthogonal initialisation with `sqrt(2)` scale (standard for ReLU) and a small output scale of `0.01` keeps Q-values close to zero initially, reducing overestimation at the start of training.


In [ ]:
class QNetwork(nn.Module):
    num_actions: int

    @nn.compact
    def __call__(self, obs):
        x = nn.relu(nn.Dense(256, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(obs))
        x = nn.relu(nn.Dense(256, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(x))
        x = nn.relu(nn.Dense(128, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(x))
        return nn.Dense(
            self.num_actions,
            kernel_init=nn.initializers.orthogonal(0.01),
        )(x)


network = QNetwork(num_actions=NUM_ACTIONS)

rng = jax.random.PRNGKey(SEED)
rng, init_key = jax.random.split(rng)

# Initialise both online and target networks with the same weights.
dummy_obs = jnp.zeros((1, OBS_DIM), dtype=jnp.float32)
online_params = network.init(init_key, dummy_obs)
target_params = online_params   # hard copy at step 0

optimizer = optax.chain(
    optax.clip_by_global_norm(10.0),
    optax.adam(LEARNING_RATE),
)
optimizer_state = optimizer.init(online_params)

q_out = network.apply(online_params, dummy_obs)
print("Q-network output shape:", q_out.shape, "  (1 sample × NUM_ACTIONS)")


## 7. DQN agent — Double DQN update

**Double DQN** decouples action *selection* from action *evaluation* to reduce overestimation bias:

1. Use the **online** network to pick the best next action: `a* = argmax_a Q_online(s', a)`
2. Use the **target** network to evaluate that action: `Q_target(s', a*)`
3. TD target: `r + γ · Q_target(s', a*) · (1 − terminal)`

The update function is JIT'd. It takes frozen `target_params` as a pure argument (not mutated inside JAX), so the hard copy in the training loop stays clean Python.


In [ ]:
@jax.jit
def update_step(
    online_params,
    target_params,
    optimizer_state,
    batch_obs,
    batch_actions,
    batch_rewards,
    batch_next_obs,
    batch_terminals,
):
    def loss_fn(params):
        # Current Q-values for the actions taken.
        q_values = network.apply(params, batch_obs)   # (B, NUM_ACTIONS)
        q_taken = q_values[jnp.arange(batch_obs.shape[0]), batch_actions]  # (B,)

        # Double DQN target.
        next_online_q = network.apply(params, batch_next_obs)          # online selects
        best_actions = jnp.argmax(next_online_q, axis=-1)             # (B,)
        next_target_q = network.apply(target_params, batch_next_obs)   # target evaluates
        next_q = next_target_q[jnp.arange(batch_obs.shape[0]), best_actions]  # (B,)

        targets = batch_rewards + GAMMA * next_q * (1.0 - batch_terminals)  # (B,)
        targets = jax.lax.stop_gradient(targets)

        loss = jnp.mean((q_taken - targets) ** 2)
        return loss

    loss, grads = jax.value_and_grad(loss_fn)(online_params)
    updates, new_optimizer_state = optimizer.update(grads, optimizer_state, online_params)
    new_params = optax.apply_updates(online_params, updates)
    return new_params, new_optimizer_state, loss


print("JIT-compiled update_step ready.")


## 8. MJX-batched collection step

All `NUM_ENVS` environments are stepped in parallel with `jax.vmap` inside a single `jax.jit` call. This is the main GPU acceleration point for DQN collection.

**Epsilon-greedy** selection is fully vectorised:
- Draw a uniform random value per environment.
- If `u < epsilon`, sample a random discrete action.
- Otherwise, take `argmax Q_online(obs)`.

**Auto-reset** is handled inside `collect_step_batch`. Environments that are `done` are reset via `data.where(done, reset_data)`, so the returned `next_data` is always a valid next-state. Importantly, the transition stored in the replay buffer uses the **pre-reset** `next_obs` (the true terminal observation), so the bootstrap target is correct.


In [ ]:
@jax.jit
def collect_step_batch(q_params, env_state, obs, epsilon, rng_key):
    """
    Step all NUM_ENVS environments by one timestep.

    Returns
    -------
    next_env_state : EnvState  — state after auto-reset
    next_obs       : (NUM_ENVS, OBS_DIM) — obs for next iteration (post-reset where done)
    obs            : (NUM_ENVS, OBS_DIM) — current obs (for buffer: s)
    actions        : (NUM_ENVS,) int32
    rewards        : (NUM_ENVS,) float32
    terminal_obs   : (NUM_ENVS, OBS_DIM) — true next obs before reset (for buffer: s')
    terminals      : (NUM_ENVS,) bool — rail hit only (not time-limit)
    dones          : (NUM_ENVS,) bool — rail hit OR time-limit
    episode_returns: (NUM_ENVS,) — episode return (valid only when done)
    episode_lengths: (NUM_ENVS,) int32
    episode_capture: (NUM_ENVS,) bool
    """
    rng_key, action_key, reset_key = jax.random.split(rng_key, 3)

    # Vectorised epsilon-greedy.
    greedy_actions = jnp.argmax(network.apply(q_params, obs), axis=-1)  # (N,)
    random_actions = jax.random.randint(action_key, (NUM_ENVS,), 0, NUM_ACTIONS)  # (N,)
    uniform_draws = jax.random.uniform(action_key, (NUM_ENVS,))
    actions = jnp.where(uniform_draws < epsilon, random_actions, greedy_actions).astype(jnp.int32)

    # Physics step (all envs in parallel).
    stepped_data = step_data_batch(env_state.data, actions)

    # Reward and termination.
    reward, terminated, x, theta1, theta2 = _compute_reward(stepped_data)
    terminal_obs = observation_from_data(stepped_data)

    # Episode bookkeeping.
    next_step_count = env_state.step_count + 1
    truncated = next_step_count >= MAX_EPISODE_STEPS
    done = terminated | truncated

    upright = (jnp.abs(theta1) < 0.3) & (jnp.abs(theta2) < 0.3)
    episode_return = env_state.episode_return + reward
    episode_length = env_state.episode_length + 1
    captured = env_state.captured_upright | upright

    # Auto-reset done environments.
    reset_keys = jax.random.split(reset_key, NUM_ENVS)
    reset_data = reset_data_batch(reset_keys)

    next_data = jax.vmap(
        lambda stepped, reset, is_done: stepped.where(is_done, reset)
    )(stepped_data, reset_data, done)

    reset_obs = observation_from_data(reset_data)
    next_obs = jnp.where(done[:, None], reset_obs, terminal_obs)

    episode_returns_out = jnp.where(done, episode_return, jnp.nan)
    episode_lengths_out = jnp.where(done, episode_length, 0)
    episode_capture_out = jnp.where(done, captured, False)

    next_env_state = EnvState(
        data=next_data,
        step_count=jnp.where(done, 0, next_step_count),
        episode_return=jnp.where(done, 0.0, episode_return),
        episode_length=jnp.where(done, 0, episode_length),
        captured_upright=jnp.where(done, False, captured),
    )

    return (
        next_env_state,
        next_obs,
        obs,
        actions,
        reward.astype(jnp.float32),
        terminal_obs,
        terminated,
        done,
        episode_returns_out,
        episode_lengths_out,
        episode_capture_out,
    )


print("collect_step_batch compiled.")


## 9. Checkpoints and metrics

Atomic writes (write to `.tmp`, then `os.replace`) prevent corrupt checkpoints on runtime disconnects. Only the 3 most-recent numbered checkpoints are kept to save Drive quota; `latest.pkl` is always present for resume.


In [ ]:
import csv
import pickle
import time
from datetime import datetime, timezone

METRIC_FIELDS = [
    "timestamp",
    "global_steps",
    "episodes",
    "mean_reward",
    "loss",
    "epsilon",
    "mean_episode_length",
    "steps_per_second",
    "capture_rate",
]


def save_checkpoint(online_params, target_params, optimizer_state, rng, global_steps):
    payload = {
        "version": 1,
        "global_steps": int(global_steps),
        "online_params": jax.device_get(online_params),
        "target_params": jax.device_get(target_params),
        "opt_state": jax.device_get(optimizer_state),
        "rng": np.asarray(jax.device_get(rng)),
    }

    final_path = CHECKPOINT_DIR / f"checkpoint_{global_steps:012d}.pkl"
    tmp_path = final_path.with_suffix(".tmp")

    with open(tmp_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, final_path)

    # Always update latest.pkl.
    latest = CHECKPOINT_DIR / "latest.pkl"
    latest_tmp = CHECKPOINT_DIR / "latest.tmp"
    with open(latest_tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(latest_tmp, latest)

    # Prune: keep only the 3 most recent numbered checkpoints.
    numbered = sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"))
    for old in numbered[:-3]:
        old.unlink(missing_ok=True)

    return final_path


def restore_checkpoint():
    global online_params, target_params, optimizer_state, rng
    latest = CHECKPOINT_DIR / "latest.pkl"
    if not latest.exists():
        print("No checkpoint found — starting fresh.")
        return 0

    with open(latest, "rb") as f:
        payload = pickle.load(f)

    online_params = jax.device_put(payload["online_params"])
    target_params = jax.device_put(payload["target_params"])
    optimizer_state = jax.device_put(payload["opt_state"])
    rng = jnp.asarray(payload["rng"], dtype=jnp.uint32)

    gs = int(payload["global_steps"])
    print(f"Restored checkpoint at global_steps={gs:,}")
    return gs


def append_metric(row: dict) -> None:
    exists = METRICS_PATH.exists()
    with open(METRICS_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=METRIC_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


## 10. Training loop

Unlike PPO's fully-compiled `lax.scan` rollout, DQN has a Python-level loop. The reasons:

1. The replay buffer `sample` and `add_batch` are on CPU numpy — they cannot live inside `jax.jit`.
2. The training frequency, warmup gate, and target copy schedule are data-dependent Python conditions.
3. The bottleneck is the network update on GPU, not the Python loop overhead.

Each iteration adds `NUM_ENVS` transitions to the buffer, so the effective sample rate is high even with a simple Python loop.

**Epsilon** decays linearly from `EPSILON_START → EPSILON_END` over `EPSILON_DECAY_STEPS` global steps.


In [ ]:
# Restore from checkpoint if available.
global_steps = restore_checkpoint()

# Fresh environment state (policy/optimizer may already be restored).
rng, env_key = jax.random.split(rng)
env_state, obs = reset_batch(jax.random.split(env_key, NUM_ENVS))

# Running episode metrics (ring buffer of recently completed episodes).
_recent_returns: list = []
_recent_lengths: list = []
_recent_captures: list = []
_total_episodes: int = 0
_last_loss: float = float("nan")

session_start = time.perf_counter()
session_start_steps = global_steps
last_log_steps = global_steps
last_ckpt_steps = global_steps

print(
    f"Training from global_steps={global_steps:,} toward {TOTAL_STEPS:,}\n"
    f"  warmup={WARMUP_STEPS:,}  train_freq={TRAIN_FREQUENCY}  "
    f"target_freq={TARGET_FREQUENCY}  buffer={BUFFER_CAPACITY:,}"
)

try:
    while global_steps < TOTAL_STEPS:
        # --- Epsilon schedule ---
        epsilon = float(np.clip(
            EPSILON_START - (global_steps / EPSILON_DECAY_STEPS) * (EPSILON_START - EPSILON_END),
            EPSILON_END,
            EPSILON_START,
        ))

        # --- Collect one step from all NUM_ENVS environments ---
        rng, step_key = jax.random.split(rng)
        (
            env_state,
            next_obs,
            cur_obs,
            actions,
            rewards,
            terminal_obs,
            terminals,
            dones,
            ep_returns,
            ep_lengths,
            ep_captures,
        ) = collect_step_batch(online_params, env_state, obs, epsilon, step_key)

        # Move to CPU numpy for buffer insertion.
        cur_obs_np = np.asarray(jax.device_get(cur_obs))
        actions_np = np.asarray(jax.device_get(actions))
        rewards_np = np.asarray(jax.device_get(rewards))
        terminal_obs_np = np.asarray(jax.device_get(terminal_obs))
        terminals_np = np.asarray(jax.device_get(terminals))

        replay_buffer.add_batch(
            cur_obs_np, actions_np, rewards_np, terminal_obs_np, terminals_np
        )

        obs = next_obs
        global_steps += NUM_ENVS

        # --- Collect episode statistics ---
        dones_np = np.asarray(jax.device_get(dones))
        ep_returns_np = np.asarray(jax.device_get(ep_returns))
        ep_lengths_np = np.asarray(jax.device_get(ep_lengths))
        ep_captures_np = np.asarray(jax.device_get(ep_captures))

        finished = dones_np.nonzero()[0]
        for i in finished:
            _recent_returns.append(float(ep_returns_np[i]))
            _recent_lengths.append(int(ep_lengths_np[i]))
            _recent_captures.append(bool(ep_captures_np[i]))
            _total_episodes += 1

        # Keep only the last 200 completed episodes for rolling stats.
        _recent_returns = _recent_returns[-200:]
        _recent_lengths = _recent_lengths[-200:]
        _recent_captures = _recent_captures[-200:]

        # --- Network update ---
        if (
            global_steps >= WARMUP_STEPS
            and global_steps % TRAIN_FREQUENCY == 0
            and replay_buffer.size >= BATCH_SIZE
        ):
            batch = replay_buffer.sample(BATCH_SIZE)
            online_params, optimizer_state, loss = update_step(
                online_params,
                target_params,
                optimizer_state,
                jnp.asarray(batch["obs"]),
                jnp.asarray(batch["actions"]),
                jnp.asarray(batch["rewards"]),
                jnp.asarray(batch["next_obs"]),
                jnp.asarray(batch["terminals"]),
            )
            _last_loss = float(jax.device_get(loss))

        # --- Hard target network copy ---
        if global_steps % TARGET_FREQUENCY == 0:
            target_params = online_params

        # --- Logging ---
        if global_steps - last_log_steps >= LOG_EVERY_STEPS:
            elapsed = time.perf_counter() - session_start
            sps = (
                (global_steps - session_start_steps) / elapsed
                if elapsed > 0 else 0.0
            )

            mean_r = float(np.mean(_recent_returns)) if _recent_returns else float("nan")
            mean_l = float(np.mean(_recent_lengths)) if _recent_lengths else float("nan")
            cap_r = float(np.mean(_recent_captures)) if _recent_captures else float("nan")

            row = {
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "global_steps": global_steps,
                "episodes": _total_episodes,
                "mean_reward": mean_r,
                "loss": _last_loss,
                "epsilon": epsilon,
                "mean_episode_length": mean_l,
                "steps_per_second": sps,
                "capture_rate": cap_r,
            }
            append_metric(row)

            print(
                f"steps={global_steps:>9,}  "
                f"eps={epsilon:.3f}  "
                f"R={mean_r:>8.2f}  "
                f"len={mean_l:>7.1f}  "
                f"cap={cap_r:>5.1%}  "
                f"loss={_last_loss:.5f}  "
                f"SPS={sps:,.0f}"
            )
            last_log_steps = global_steps

        # --- Checkpoint ---
        if global_steps - last_ckpt_steps >= CHECKPOINT_EVERY_STEPS:
            path_ckpt = save_checkpoint(
                online_params, target_params, optimizer_state, rng, global_steps
            )
            print("checkpoint:", path_ckpt)
            last_ckpt_steps = global_steps

except KeyboardInterrupt:
    path_ckpt = save_checkpoint(
        online_params, target_params, optimizer_state, rng, global_steps
    )
    print("\nInterrupted safely. checkpoint:", path_ckpt)
    raise

# Final checkpoint.
path_ckpt = save_checkpoint(
    online_params, target_params, optimizer_state, rng, global_steps
)
print("Training complete. Final checkpoint:", path_ckpt)


## 11. Training dashboard

Six subplots in a 2×3 grid: episode return, capture rate, episode length, Q-network loss, epsilon schedule, and steps-per-second throughput.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if not METRICS_PATH.exists():
    raise FileNotFoundError("No metrics.csv yet. Run training first.")

metrics = (
    pd.read_csv(METRICS_PATH)
    .drop_duplicates("global_steps", keep="last")
    .sort_values("global_steps")
)

window = min(20, len(metrics))
x = metrics["global_steps"]

figure, axes = plt.subplots(
    2, 3,
    figsize=(16, 9),
    constrained_layout=True,
)
figure.suptitle(
    "TIPy Double Pendulum — DQN + MJX GPU",
    fontsize=16,
    fontweight="bold",
)


def smooth(series):
    return series.rolling(window, min_periods=1).mean()


axes[0, 0].plot(x, metrics["mean_reward"], alpha=0.25)
axes[0, 0].plot(x, smooth(metrics["mean_reward"]))
axes[0, 0].set_title("Episode return")

axes[0, 1].plot(x, metrics["capture_rate"])
axes[0, 1].set_title("Both-upright capture rate")
axes[0, 1].set_ylim(0, 1)

axes[0, 2].plot(x, metrics["mean_episode_length"])
axes[0, 2].set_title("Episode length")

axes[1, 0].plot(x, metrics["loss"], alpha=0.5)
axes[1, 0].plot(x, smooth(metrics["loss"]))
axes[1, 0].set_title("Q-network loss (MSE TD error)")

axes[1, 1].plot(x, metrics["epsilon"])
axes[1, 1].set_title("Epsilon (exploration rate)")
axes[1, 1].set_ylim(0, 1.05)

axes[1, 2].plot(x, metrics["steps_per_second"])
axes[1, 2].set_title("Environment steps / second")

for ax in axes.flat:
    ax.grid(alpha=0.2)
    ax.set_xlabel("global steps")

figure.savefig(DASHBOARD_PATH, dpi=160)
plt.show()

print("saved:", DASHBOARD_PATH)


## 12. Greedy evaluation

Run 64 deterministic episodes with `epsilon = 0`. The Q-network selects `argmax Q(s, a)` at every step. Evaluation uses the same batched MJX collection path.


In [ ]:
EVAL_ENVS = 64

@jax.jit
def greedy_collect_step(eval_params, eval_data, eval_obs, rng_key):
    """One greedy step over EVAL_ENVS environments (epsilon = 0)."""
    greedy_actions = jnp.argmax(network.apply(eval_params, eval_obs), axis=-1)
    stepped_data = step_data_batch(eval_data, greedy_actions)
    reward, terminated, x, theta1, theta2 = _compute_reward(stepped_data)
    next_obs = observation_from_data(stepped_data)
    upright = (jnp.abs(theta1) < 0.3) & (jnp.abs(theta2) < 0.3)
    return stepped_data, next_obs, greedy_actions, reward, terminated, upright


rng, eval_key = jax.random.split(rng)
eval_keys = jax.random.split(eval_key, EVAL_ENVS)
eval_data = reset_data_batch(eval_keys)
eval_obs = observation_from_data(eval_data)

eval_returns = np.zeros(EVAL_ENVS, dtype=np.float32)
eval_lengths = np.zeros(EVAL_ENVS, dtype=np.int32)
eval_captured = np.zeros(EVAL_ENVS, dtype=bool)
eval_done = np.zeros(EVAL_ENVS, dtype=bool)

print("Running greedy evaluation...")
for _ in range(MAX_EPISODE_STEPS):
    if eval_done.all():
        break

    rng, eval_step_key = jax.random.split(rng)
    eval_data, eval_obs, _, rewards, terminated, upright = greedy_collect_step(
        online_params, eval_data, eval_obs, eval_step_key
    )

    rewards_np = np.asarray(jax.device_get(rewards))
    terminated_np = np.asarray(jax.device_get(terminated))
    upright_np = np.asarray(jax.device_get(upright))

    active = ~eval_done
    eval_returns += np.where(active, rewards_np, 0.0)
    eval_lengths += active.astype(np.int32)
    eval_captured |= active & upright_np
    eval_done |= terminated_np
    eval_done |= eval_lengths >= MAX_EPISODE_STEPS

print(f"Evaluation over {EVAL_ENVS} episodes:")
print(f"  mean return        : {eval_returns.mean():.2f}")
print(f"  mean episode length: {eval_lengths.mean():.1f}")
print(f"  capture rate       : {eval_captured.mean():.1%}")


## 13. Replay video

Training and evaluation are MJX GPU. For the MP4, this cell intentionally uses the CPU `mujoco.Renderer` — rendering a single world one frame at a time is exactly where MJX-JAX offers no advantage, and the CPU renderer is far simpler to use.

The policy runs on GPU (JAX) and the rendered frame is pulled to CPU only for encoding.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED = 50_004
REPLAY_SECONDS = 10.0
REPLAY_FPS = 50

# CPU MuJoCo — only used as a renderer for one visual replay episode.
replay_model = mujoco.MjModel.from_xml_string(MODEL_XML)
replay_data = mujoco.MjData(replay_model)

rng_np = np.random.default_rng(REPLAY_SEED)
absolute1 = np.pi + rng_np.uniform(-0.05, 0.05)
absolute2 = np.pi + rng_np.uniform(-0.05, 0.05)

replay_data.qpos[:] = [
    rng_np.uniform(-0.05, 0.05),
    absolute1,
    absolute2 - absolute1,
]

absolute_speed1 = rng_np.uniform(-0.05, 0.05)
absolute_speed2 = rng_np.uniform(-0.05, 0.05)

replay_data.qvel[:] = [
    rng_np.uniform(-0.05, 0.05),
    absolute_speed1,
    absolute_speed2 - absolute_speed1,
]

mujoco.mj_forward(replay_model, replay_data)

renderer = mujoco.Renderer(replay_model, height=480, width=640)

writer = imageio.get_writer(
    REPLAY_PATH,
    fps=REPLAY_FPS,
    codec="libx264",
    quality=8,
)

frame_stride = max(1, round(1.0 / (replay_model.opt.timestep * REPLAY_FPS)))


def cpu_obs(data):
    """Build the 8-D observation from CPU MjData."""
    x = float(data.qpos[0])
    relative1 = float(data.qpos[1])
    relative2 = float(data.qpos[2])
    dx = float(data.qvel[0])
    w1 = float(data.qvel[1])
    w2rel = float(data.qvel[2])

    theta1 = (relative1 + np.pi) % (2 * np.pi) - np.pi
    theta2 = (relative1 + relative2 + np.pi) % (2 * np.pi) - np.pi

    return np.array(
        [
            np.clip(x, -RAIL_LIMIT, RAIL_LIMIT),
            np.clip(dx / 3.0, -1.0, 1.0),
            np.cos(theta1),
            np.sin(theta1),
            np.cos(theta2),
            np.sin(theta2),
            np.clip(w1 / 15.0, -1.0, 1.0),
            np.clip((w1 + w2rel) / 15.0, -1.0, 1.0),
        ],
        dtype=np.float32,
    )


num_steps = int(REPLAY_SECONDS / replay_model.opt.timestep)

for step in range(num_steps):
    obs_cpu = cpu_obs(replay_data)

    # Greedy action from GPU Q-network.
    q_vals = network.apply(online_params, jnp.asarray(obs_cpu)[None, :])  # (1, NUM_ACTIONS)
    action_idx = int(jax.device_get(jnp.argmax(q_vals, axis=-1))[0])
    force = float(ACTION_TABLE[action_idx])

    replay_data.ctrl[0] = force
    mujoco.mj_step(replay_model, replay_data)

    if step % frame_stride == 0:
        renderer.update_scene(replay_data, camera="replay")
        writer.append_data(renderer.render())

    if abs(float(replay_data.qpos[0])) >= RAIL_LIMIT:
        print(f"Cart hit rail limit at step {step}.")
        break

writer.close()
renderer.close()

print("saved:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
